# 00. Od potrzeby badawczej do procedury

[![Otwórz w Colabie](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caqdastm/ai_qda-workshop-1u/blob/main/00_github_colab/00_start_here_github_colab.ipynb)

**Odznaka otwiera wzorzec** z repozytorium prowadzących. Do trwałej pracy utwórz prywatne repo przez **Use this template** i otwieraj notebook z własnego repo.

Ten notebook nie jest lekcją Pythona. Celem jest rozpoznanie
procedury potrzebnej w pracy z materiałem i opisanie jej tak, aby
asystent AI mógł później przygotować małą implementację.


## Rytm pracy

**Potrzeba badawcza -> procedura -> kontrakt -> wykonanie -> obserwacja
wyniku -> decyzja badacza.**

Kod będzie przygotowaną infrastrukturą wykonującą procedurę. Nie
musisz zapamiętywać nazw zmiennych ani składni. W tym notebooku
edytujesz pola formularza i interpretujesz rezultat.


In [42]:
# @title Sprawdź prywatny workspace — uruchom bez edycji { display-mode: "form" }
from pathlib import Path
import base64
import os
import re
import subprocess
import sys

PUBLIC_REPOSITORY_URL = "https://github.com/caqdastm/ai_qda-workshop-1u.git"
PUBLIC_REPOSITORY_SLUG = "caqdastm/ai_qda-workshop-1u"
REPOSITORY_REF = "main"

def _colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

PARTICIPANT_REPOSITORY = str(
    _colab_secret("AI_QDA_REPOSITORY") or ""
).strip().rstrip("/")
if PARTICIPANT_REPOSITORY.endswith(".git"):
    PARTICIPANT_REPOSITORY = PARTICIPANT_REPOSITORY[:-4]
if PARTICIPANT_REPOSITORY.startswith("https://github.com/"):
    PARTICIPANT_REPOSITORY = PARTICIPANT_REPOSITORY.removeprefix(
        "https://github.com/"
    )
if PARTICIPANT_REPOSITORY and not re.fullmatch(
    r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", PARTICIPANT_REPOSITORY
):
    raise ValueError(
        "Sekret AI_QDA_REPOSITORY podaj jako login/nazwa-repozytorium."
    )
if PARTICIPANT_REPOSITORY.lower() == PUBLIC_REPOSITORY_SLUG.lower():
    raise ValueError(
        "AI_QDA_REPOSITORY musi wskazywać Twoje prywatne repo, "
        "nie repo prowadzących."
    )

GITHUB_TOKEN = _colab_secret("GITHUB_TOKEN")
if PARTICIPANT_REPOSITORY:
    if not GITHUB_TOKEN:
        raise RuntimeError(
            "Włącz dla tego notebooka dostęp do sekretu GITHUB_TOKEN."
        )
    REPOSITORY_URL = f"https://github.com/{PARTICIPANT_REPOSITORY}.git"
    WORKSPACE_MODE = "participant_repository"
else:
    REPOSITORY_URL = PUBLIC_REPOSITORY_URL
    WORKSPACE_MODE = "public_demo"

REPO_ROOT = Path("/content/ai_qda_workshop_workspace")
if not (REPO_ROOT / "04_vibe_coding" / "workshop_support.py").is_file():
    clone_environment = os.environ.copy()
    if GITHUB_TOKEN:
        encoded = base64.b64encode(
            f"x-access-token:{GITHUB_TOKEN}".encode("utf-8")
        ).decode("ascii")
        clone_environment["GIT_CONFIG_COUNT"] = "1"
        clone_environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
        clone_environment["GIT_CONFIG_VALUE_0"] = (
            f"AUTHORIZATION: basic {encoded}"
        )
    clone_result = subprocess.run(
        [
            "git", "clone", "--depth", "1", "--branch", REPOSITORY_REF,
            REPOSITORY_URL, str(REPO_ROOT),
        ],
        env=clone_environment,
        capture_output=True,
        text=True,
    )
    if clone_result.returncode != 0:
        raise RuntimeError(
            "Nie udało się otworzyć repozytorium roboczego. Sprawdź "
            "sekrety AI_QDA_REPOSITORY i GITHUB_TOKEN oraz dostęp tokenu "
            f"do repo. Git: {clone_result.stderr.strip()}"
        )

support_dir = REPO_ROOT / "04_vibe_coding"
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))
from workshop_support import (
    publish_outputs_to_github,
    read_secret,
    save_dataframe,
    save_json,
)

INTRO_OUTPUTS_DIR = REPO_ROOT / "00_github_colab" / "outputs"
INTRO_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print("Tryb workspace:", WORKSPACE_MODE)
print("Repo uczestnika:", PARTICIPANT_REPOSITORY or "brak — publiczny tryb demonstracyjny")
print("Wyniki wprowadzenia:", INTRO_OUTPUTS_DIR)


Tryb workspace: participant_repository
Repo uczestnika: magdalena-netizen/MKK-Szkola-letnia
Wyniki wprowadzenia: /content/ai_qda_workshop_workspace/00_github_colab/outputs


## 1. Najpierw nazwij potrzebę

Przeczytaj trzy sytuacje. W każdej potrzebna jest inna procedura:

1. Po podziale transkrypcji chcesz zachować dokładny tekst, kolejność,
   mówcę i możliwość powrotu do źródła.
2. Niektórych etykiet mówców nie da się rozstrzygnąć bez kontekstu;
   chcesz skierować je do przeglądu badacza.
3. Masz wiele wywiadów i chcesz widzieć, które przeszły dany etap, a
   które nadal czekają.

Porozmawiaj w parze: **jakie działanie powinno być powtarzalne i jaki
rezultat musi być widoczny?** Nie proponuj jeszcze kodu.

Dalej pracujemy na pierwszej sytuacji, ale ta sama karta pasuje do
dwóch pozostałych.


In [43]:
# @title 2. Karta procedury — edytuj tylko pola formularza { display-mode: "form" }
problem_do_rozwiazania = "Podział rozmowy może zerwać związek jednostki z kolejnością i źródłem." # @param {type:"string"}
material_wejsciowy = "Krótka transkrypcja z identyfikatorem wywiadu, etykietą mówcy i dokładnym tekstem." # @param {type:"string"}
widoczny_rezultat = "Tabela jednostek z rolą mówcy, stabilnym ID, kolejnością i niezmienionym tekstem." # @param {type:"string"}
kontrola_automatyczna = "Liczba i kolejność wierszy oraz tekst pozostają takie same; ID są unikalne." # @param {type:"string"}
decyzja_badacza = "Czy granice jednostek są użyteczne analitycznie i jak rozstrzygnąć niejednoznaczne etykiety." # @param {type:"string"}

PROCEDURE_CARD = {
    "problem": problem_do_rozwiazania,
    "input": material_wejsciowy,
    "observable_result": widoczny_rezultat,
    "automatic_check": kontrola_automatyczna,
    "researcher_decision": decyzja_badacza,
}


In [44]:
# @title 3. Infrastruktura: pokaż i sprawdź kartę { display-mode: "form" }
import pandas as pd
from IPython.display import display

labels = {
    "problem": "Problem w pracy z materiałem",
    "input": "Materiał wejściowy",
    "observable_result": "Widoczny rezultat",
    "automatic_check": "Kontrola techniczna",
    "researcher_decision": "Decyzja badacza",
}
card_table = pd.DataFrame(
    [{"element": labels[key], "opis": value} for key, value in PROCEDURE_CARD.items()]
)
display(card_table)

missing = [labels[key] for key, value in PROCEDURE_CARD.items() if not value.strip()]
if missing:
    print("Uzupełnij pola:", ", ".join(missing))
else:
    print("Karta jest kompletna. Można przekazać ją do etapu implementacji.")
    print("Kontrola techniczna nie zastępuje decyzji badacza.")


,element,opis
0,Problem w pracy z materiałem,Podział rozmowy może zerwać związek jednostki ...
1,Materiał wejściowy,"Krótka transkrypcja z identyfikatorem wywiadu,..."
2,Widoczny rezultat,"Tabela jednostek z rolą mówcy, stabilnym ID, k..."
3,Kontrola techniczna,Liczba i kolejność wierszy oraz tekst pozostaj...
4,Decyzja badacza,Czy granice jednostek są użyteczne analityczni...


Karta jest kompletna. Można przekazać ją do etapu implementacji.
Kontrola techniczna nie zastępuje decyzji badacza.


## 4. Publiczny wzorzec i prywatny workspace

- **Publiczne repo** zawiera czysty wzorzec materiałów.
- **Prywatne repo z szablonu** jest Twoim trwałym workspace.
- **Runtime Colaba** jest tymczasowy i znika po zakończeniu sesji.
- **Plik w `outputs/`** jest produktem wykonania procedury.
- **Commit i push** zapisują sprawdzony produkt w historii prywatnego repo.

Notebook używa sekretów `AI_QDA_REPOSITORY` i `GITHUB_TOKEN`. Nie
umieszcza tokenu w adresie repo, kodzie, pliku ani commicie.


In [45]:
# @title 5. Infrastruktura: zapisz kartę jako artefakt { display-mode: "form" }
output_path = INTRO_OUTPUTS_DIR / "00_procedure_card.csv"
save_dataframe(output_path, card_table)
print("Zapisano:", output_path)
print("Najpierw przeczytaj plik, dopiero potem utwórz commit.")


Zapisano: /content/ai_qda_workshop_workspace/00_github_colab/outputs/00_procedure_card.csv
Najpierw przeczytaj plik, dopiero potem utwórz commit.


In [67]:
# @title Zapisz sprawdzone wyniki w swoim repo GitHub { display-mode: "form" }
PUBLISH_RESULTS_TO_GITHUB = True # @param {type:"boolean"}

if PUBLISH_RESULTS_TO_GITHUB:
    if WORKSPACE_MODE != "participant_repository":
        raise RuntimeError(
            "Dodaj w Colab Secrets AI_QDA_REPOSITORY i GITHUB_TOKEN, "
            "włącz ich dostęp i uruchom notebook ponownie od początku."
        )
    publication = publish_outputs_to_github(
        REPO_ROOT,
        [INTRO_OUTPUTS_DIR],
        message='AI QDA: sprawdź prywatny workspace',
        participant_repository=PARTICIPANT_REPOSITORY,
        token=read_secret("GITHUB_TOKEN"),
        branch=REPOSITORY_REF,
        include_api_logs=False,
    )
    print("Zapis GitHub:", publication["status"])
    print("Commit:", publication.get("commit", "bez nowej zmiany"))
    print("Pliki:", publication["paths"])
else:
    print(
        "Wyniki są tylko w runtime. Po kontroli ustaw "
        "PUBLISH_RESULTS_TO_GITHUB=True i uruchom komórkę ponownie."
    )


RuntimeError: Operacja Git nie powiodła się. Sprawdź nazwę repozytorium, dostęp sekretu GITHUB_TOKEN i uprawnienie Contents: Read and write. Szczegóły Git: remote: Permission to magdalena-netizen/MKK-Szkola-letnia.git denied to magdalena-netizen.
fatal: unable to access 'https://github.com/magdalena-netizen/MKK-Szkola-letnia.git/': The requested URL returned error: 403

## 6. Zapis pracy

Po sprawdzeniu karty ustaw `PUBLISH_RESULTS_TO_GITHUB=True`. Oczekiwany
status to `pushed`; ponowne uruchomienie bez zmian może zwrócić
`up_to_date`. Na stronie prywatnego repo znajdź plik
`00_github_colab/outputs/00_procedure_card.csv` i commit.

Sam notebook zapisz osobno przez **Plik → Zapisz kopię w GitHubie**,
wybierając własne prywatne repo i tę samą ścieżkę pliku.

**Exit ticket:** jednym zdaniem nazwij procedurę, którą chcesz umieć
zaprojektować podczas głównej części Vibe Coding.
